# Echogram vertical coordinates and geometry

This notebook uses one EK80 RAW file to compare sample indices, physical range, depth, MVBS, and reference-channel resampling. The 38 kHz channel is displayed throughout with the same colour limits.

The RAW file is the same one used in the [Echopype Getting Started notebook](https://echopype-examples.readthedocs.io/en/latest/getting_started.html).


In [ ]:
import echopype as ep
import echopype.colormap
import echoshader
import holoviews as hv
import matplotlib as mpl
import panel as pn
from echopype.commongrid import resample_to_geometry

In [2]:
pn.extension()
hv.extension("bokeh")

In [3]:
ek500_cmap = mpl.colormaps["ep.ek500"]

plot_options = dict(cmap=ek500_cmap, clim=(-80, -30), colorbar=True,
                    width=1000, height=350)

## Open and calibrate one RAW file

Convert the EK80 file and calibrate its CW power data to volume backscattering strength (`Sv`). Select 38 kHz for display and 200 kHz as the reference geometry.

In [4]:
raw_path = (
    "s3://noaa-wcsd-pds/data/raw/Bell_M._Shimada/SH2306/EK80/"
    "Hake-D20230811-T165727.raw"
)
ed = ep.open_raw(raw_path, sonar_model="EK80", storage_options={"anon": True})
ds_Sv = ep.calibrate.compute_Sv(ed, waveform_mode="CW", encode_mode="power")

selected_channel = ds_Sv.channel.sel(channel=ds_Sv.frequency_nominal == 38000).item()
reference_channel = ds_Sv.channel.sel(channel=ds_Sv.frequency_nominal == 200000).item()
ds_Sv


parsing file Hake-D20230811-T165727.raw, time of first ping: 2023-Aug-11 16:57:27


<xarray.Dataset> Size: 617MB
Dimensions:                        (channel: 5, ping_time: 213,
                                    range_sample: 36198, filenames: 1)
Coordinates:
  * channel                        (channel) object 40B 'WBT 400140-15 ES120-...
  * ping_time                      (ping_time) datetime64[ns] 2kB 2023-08-11T...
  * range_sample                   (range_sample) int64 290kB 0 1 ... 36197
  * filenames                      (filenames) int64 8B 0
Data variables: (12/25)
    Sv                             (channel, ping_time, range_sample) float64 308MB ...
    echo_range                     (channel, ping_time, range_sample) float64 308MB ...
    tau_effective                  (channel) float64 40B 0.0009239 ... 0.0009306
    frequency_nominal              (channel) float64 40B 1.2e+05 ... 2e+05
    sound_speed                    float64 8B 1.48e+03
    sound_absorption               (channel) float64 40B 0.03281 ... 0.0463
    ...                             ...
    beamwidth_athwartship          (channel) float64 40B 6.74 10.6 ... 6.81 6.07
    impedance_transducer           (channel) float64 40B 75.0 75.0 ... 75.0 75.0
    impedance_transceiver          (channel) int64 40B 5400 5400 5400 5400 5400
    receiver_sampling_frequency    (channel) float64 40B 1.5e+06 ... 1.5e+06
    source_filenames               (filenames) <U82 328B 's3://noaa-wcsd-pds/...
    water_level                    float64 8B 0.0
Attributes:
    processing_software_name:     echopype
    processing_software_version:  0.12.0
    processing_time:              2026-10-29T22:43:00+00:00
    processing_function:          calibrate.compute_Sv

> **Demonstration data**: The next block subsamples the data to at most 150 pings and 500 range samples per channel, reducing rendering time and saved notebook size. Comment out this block to use the full-resolution data. Subsequent plots, MVBS calculations, and geometry matching use whichever resolution you select.

In [5]:
import math
import warnings

warnings.warn(
    "Subsampling for compact visualization examples. Subsequent MVBS and "
    "geometry matching use reduced data, not the full-resolution observations.",
    UserWarning,
)

ds_Sv = ds_Sv.isel(
    ping_time=slice(None, None, max(1, math.ceil(ds_Sv.sizes["ping_time"] / 150))),
    range_sample=slice(None, None, max(1, math.ceil(ds_Sv.sizes["range_sample"] / 500))),
)
ds_Sv


C:\Users\lloyd\AppData\Local\Temp\ipykernel_5576\2709207933.py:4: UserWarning: Subsampling for compact visualization examples. Subsequent MVBS and geometry matching use reduced data, not the full-resolution observations.
  warnings.warn(


<xarray.Dataset> Size: 4MB
Dimensions:                        (channel: 5, ping_time: 107,
                                    range_sample: 496, filenames: 1)
Coordinates:
  * channel                        (channel) object 40B 'WBT 400140-15 ES120-...
  * ping_time                      (ping_time) datetime64[ns] 856B 2023-08-11...
  * range_sample                   (range_sample) int64 4kB 0 73 ... 36062 36135
  * filenames                      (filenames) int64 8B 0
Data variables: (12/25)
    Sv                             (channel, ping_time, range_sample) float64 2MB ...
    echo_range                     (channel, ping_time, range_sample) float64 2MB ...
    tau_effective                  (channel) float64 40B 0.0009239 ... 0.0009306
    frequency_nominal              (channel) float64 40B 1.2e+05 ... 2e+05
    sound_speed                    float64 8B 1.48e+03
    sound_absorption               (channel) float64 40B 0.03281 ... 0.0463
    ...                             ...
    beamwidth_athwartship          (channel) float64 40B 6.74 10.6 ... 6.81 6.07
    impedance_transducer           (channel) float64 40B 75.0 75.0 ... 75.0 75.0
    impedance_transceiver          (channel) int64 40B 5400 5400 5400 5400 5400
    receiver_sampling_frequency    (channel) float64 40B 1.5e+06 ... 1.5e+06
    source_filenames               (filenames) <U82 328B 's3://noaa-wcsd-pds/...
    water_level                    float64 8B 0.0
Attributes:
    processing_software_name:     echopype
    processing_software_version:  0.12.0
    processing_time:              2026-10-29T22:43:00+00:00
    processing_function:          calibrate.compute_Sv

## 1. Sample index: `range_sample`

The vertical coordinate is the instrument sample index, not a distance in metres.

In [6]:
ds_Sv.eshader.echogram(channel=selected_channel, vert_dim="range_sample").opts(**plot_options)

:QuadMesh   [ping_time,range_sample]   (Sv)

## 2. Physical range: `echo_range`

Use the calculated distance from the transducer in metres. Only the displayed coordinate changes; the Sv values are the same.

In [7]:
ds_Sv.eshader.echogram(channel=selected_channel, vert_dim="echo_range").opts(**plot_options)

:QuadMesh   [ping_time,echo_range]   (Sv)

## 3. Physical depth: `depth`

Add depth using a 9.8 m transducer-depth offset with `add_depth()`. It adds a coordinate.

In [8]:
ds_depth = ep.consolidate.add_depth(ds_Sv, ed, depth_offset=9.8)
ds_depth.eshader.echogram(channel=selected_channel, vert_dim="depth").opts(**plot_options)

xarray Dataset does not contain valid location data. Processing level attributes will not be added.


:QuadMesh   [ping_time,depth]   (Sv)

## 4. MVBS on a regular depth grid

Average Sv in linear units into 5-second × 1-metre bins, then express the result in dB. Unlike the coordinate changes above, this step aggregates observations. The output has a shared one-dimensional `depth` coordinate.

In [9]:
ds_MVBS = ep.commongrid.compute_MVBS(
    ds_depth,
    range_var="depth",
    range_bin="1m",
    ping_time_bin="5s",
    range_var_max="500m",
)

ds_MVBS.eshader.echogram(channel=selected_channel, vert_dim="depth").opts(**plot_options)

xarray Dataset does not contain valid location data. Processing level attributes will not be added.


c:\Users\lloyd\miniforge3\envs\echoshader-dev313\Lib\site-packages\echopype\commongrid\utils.py:604: UserWarning: The ```depth``` coordinate array contain NaNs. Aggregation may be negatively impacted since Flox will not aggregate any ```Sv``` values that have corresponding NaN coordinate values. Consider handling these values before calling your intended commongrid function.
  warnings.warn(


:Image   [ping_time,depth]   (Sv)

## 5. Match the reference channel's range geometry

Starting again from the subsampled calibrated Sv, resample the channels to the 200 kHz reference channel's physical range grid. Display the resampled 38 kHz data against `echo_range`. This step aligns sampling geometry; it does not convert range into depth.

See the [Echopype resampling notebook](https://echopype-examples.readthedocs.io/en/latest/resample_to_geometry.html) for additional geometry options.

In [10]:
aligned_ds = resample_to_geometry(
    ds_Sv.chunk({"channel": 1, "ping_time": 1000, "range_sample": -1}),
    target_variable="Sv",
    is_log=True,
    target_channel=reference_channel,
)

aligned_ds.eshader.echogram(channel=selected_channel, vert_dim="echo_range").opts(**plot_options)

:QuadMesh   [ping_time,echo_range]   (Sv)

end of notebook

In [11]:
from datetime import datetime, timezone
import echopype
import echoshader
import xarray
import panel
import holoviews

print(f"echopype: {echopype.__version__}")
print(f"echoshader: {echoshader.__version__}")
print(f"xarray: {xarray.__version__}")
print(f"panel: {panel.__version__}")
print(f"holoviews: {holoviews.__version__}")
print(f"Executed: {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}")

echopype: 0.12.0
echoshader: 0.1.1.dev145+gf11928cad
xarray: 2026.9.0
panel: 1.9.4
holoviews: 1.23.2
Executed: 2026-10-29 22:43 UTC
